# Chapter 12 — Model ordered RLGC lines

Source candidate · CONVERGING · checkpoint-bound evidence

> **Source candidate / CONVERGING.** Authoring figures are genuine
> exports bound to the corresponding source checkpoints, not evidence
> that every cell or numerical request has executed. The website runs no
> kernels or solvers; the generated Notebook remains zero-output.

A coupled feedline needs conductor order and matrix-valued per-length
data, not a scalar wire approximation. This Chapter authors that
physical line before separating its authoring projection from a compiled
finite-pi expansion.

## Lesson 12.1 — Author a coupled N=2 line

### Declare N=2 RLGC metadata

`reference_conductor="ground"` is RLGC metadata and legend information,
never a public `PinRef`. An N=1 line has two public signal pins; an N\>1
line has exactly 2N ordered public signal pins.

In [ ]:
from scnsim import (
    CircuitDiagramSpec,
    CircuitPlan,
    RLGC,
    SCNSimValidationError,
    SchematicLayout,
    components,
    units as u,
)
from IPython.display import display

rlgc = RLGC(
    conductors=("readout", "filter"),
    reference_conductor="ground",
    resistance_per_length=[
        [0.18, 0.0],
        [0.0, 0.22],
    ]
    * u.ohm
    / u.m,
    inductance_per_length=[
        [420.0, 75.0],
        [75.0, 395.0],
    ]
    * u.nH
    / u.m,
    conductance_per_length=[
        [0.0, 0.0],
        [0.0, 0.0],
    ]
    * u.S
    / u.m,
    capacitance_per_length=[
        [175.0, -22.0],
        [-22.0, 168.0],
    ]
    * u.pF
    / u.m,
)

`rlgc` is the metadata object consumed by the native multi-terminal
body; its conductor order determines the signal pins named next.

The conductor order is `(readout, filter)`. Resistance is diagonal;
inductance is symmetric with +75 nH/m mutual entries; capacitance has
-22 pF/m off-diagonals; and the conductance matrix is exactly zero.

### Add the one native multi-terminal line body

In [ ]:
plan = CircuitPlan(id="coupled_line")
line = plan.add(
    components.transmission_line(
        id="coupled",
        length=1.6 * u.mm,
        rlgc=rlgc,
        n_sections=8,
    )
)

`line` is the one N=2 body. Its four signal pins are named explicitly
before one conductor is placed and the other is bound.

### Name every ordered signal pin and root bus

Only the four signal pins are electrical handles. There is no reference
pin to select or ground.

In [ ]:
readout_head_pin = line.pin("head", conductor="readout")
readout_tail_pin = line.pin("tail", conductor="readout")
filter_head_pin = line.pin("head", conductor="filter")
filter_tail_pin = line.pin("tail", conductor="filter")

readout_head = plan.bus(id="readout_head")
readout_tail = plan.bus(id="readout_tail")
filter_head = plan.bus(id="filter_head")
filter_tail = plan.bus(id="filter_tail")

The four pin handles and four root buses preserve the declared head/tail
and conductor order for the following structural operations.

### Use the complete occurrence once and bind the remaining pins

The whole multi-terminal transmission-line occurrence is used exactly
once. `between(readout_head_pin, readout_tail_pin)` supplies only the
ordered endpoint pair for that structural use. It does not extract or
prefer a conductor, reduce the full RLGC body, or clone it; the
remaining conductor pins bind explicitly.

In [ ]:
readout_conductor = plan.series(
    id="readout_conductor",
    start=readout_head,
    elements=(
        line.between(readout_head_pin, readout_tail_pin),
    ),
    end=readout_tail,
)
plan.link(
    id="filter_head_binding",
    endpoints=(filter_head, filter_head_pin),
)
plan.link(
    id="filter_tail_binding",
    endpoints=(filter_tail, filter_tail_pin),
)

`readout_conductor` uses the whole multi-terminal occurrence exactly
once; its `.between(...)` call supplies the ordered endpoint pair only.
The two filter links bind every remaining public signal pin without
inventing a reference pin or reducing the full RLGC body.

The compiled N-section declaration stamps the full RLGC matrix. Each
finite pi section places its shunt contribution as half-shunts at its
two section ends; it does not reduce the coupled line to scalar or
equal-current behavior.

### Review authoring and compiled drawings separately

In [ ]:
authoring = plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        show_parameter_values=True,
    )
)
authoring.show()

The authoring drawing shows the declared body and bindings. Render the
compiled expansion separately so its finite-section detail is not
mistaken for authoring.

In [ ]:
expanded = None
try:
    expanded = plan.render_schematic(
        CircuitDiagramSpec(
            representation="compiled",
            show_parameter_values=True,
        )
    )
except SCNSimValidationError as error:
    if error.stage != "schematic_layout":
        raise
    display({
        "compiled schematic layout unavailable": str(error),
        "stage": error.stage,
        "evidence": dict(error.evidence),
    })

`expanded` is the compiled presentation when the fixed layout is
available. An unavailable compiled layout reports only its
`schematic_layout` failure; other errors still propagate and the later
authoring cells remain independent.

In [ ]:
if expanded is not None:
    display(expanded.show())

In [ ]:
if expanded is not None:
    display(expanded.audit.show())

When available, the compiled picture shows complete topology, sections,
and conductor order. Its audit table carries matrix, provenance, and
exact-zero evidence.

### Bind an independent complete N=2 body

The next Plan uses the same known `rlgc` value but declares a new native
body; it does not clone `line` or reduce it to scalar conductors. This
is the independent-peer form: every ordered signal pin binds explicitly,
so no `.between()` structural use is required. The two complete
declarations can represent the same electrical model while retaining
different authored structures and therefore potentially different Plan
identities.

In [ ]:
independent_plan = CircuitPlan(id="independent_coupled_line")
independent_line = independent_plan.add(
    components.transmission_line(
        id="coupled",
        length=1.6 * u.mm,
        rlgc=rlgc,
        n_sections=8,
    )
)
independent_readout_head_pin = independent_line.pin(
    "head", conductor="readout"
)
independent_readout_tail_pin = independent_line.pin(
    "tail", conductor="readout"
)
independent_filter_head_pin = independent_line.pin(
    "head", conductor="filter"
)
independent_filter_tail_pin = independent_line.pin(
    "tail", conductor="filter"
)
independent_readout_head = independent_plan.bus(id="readout_head")
independent_readout_tail = independent_plan.bus(id="readout_tail")
independent_filter_head = independent_plan.bus(id="filter_head")
independent_filter_tail = independent_plan.bus(id="filter_tail")

In [ ]:
independent_plan.link(
    id="readout_head_binding",
    endpoints=(independent_readout_head, independent_readout_head_pin),
)
independent_plan.link(
    id="readout_tail_binding",
    endpoints=(independent_readout_tail, independent_readout_tail_pin),
)
independent_plan.link(
    id="filter_head_binding",
    endpoints=(independent_filter_head, independent_filter_head_pin),
)
independent_plan.link(
    id="filter_tail_binding",
    endpoints=(independent_filter_tail, independent_filter_tail_pin),
)

The four explicit `PinRef`/`BusRef` bindings complete this separate
multi-terminal occurrence without added loads or an inferred priority
between conductors.

In [ ]:
independent_authoring = independent_plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        show_parameter_values=True,
    )
)

In [ ]:
independent_authoring.show()

In [ ]:
independent_authoring.audit.show()

## Lesson 12.2 — Build a tapped feedline

### Declare a standalone tapped N=1 feedline

The feedline is a separate Plan and subsystem. Its `reference_conductor`
remains metadata only, with no reference pin or grounding operation.

In [ ]:
feedline_plan = CircuitPlan(id="tapped_feedline")
feedline = feedline_plan.subsystem(id="feedline")
cpw = RLGC(
    conductors=("signal",),
    reference_conductor="ground",
    resistance_per_length=[[0.0]] * u.ohm / u.m,
    inductance_per_length=[[420.0]] * u.nH / u.m,
    conductance_per_length=[[0.0]] * u.S / u.m,
    capacitance_per_length=[[175.0]] * u.pF / u.m,
)

`feedline` owns a separate N=1 declaration, while `cpw` supplies
metadata to both independent line bodies added next.

### Add the two independent N=1 line bodies

In [ ]:
left = feedline.add(
    components.transmission_line(
        id="left",
        length=1.0 * u.mm,
        rlgc=cpw,
        n_sections=1,
    )
)
right = feedline.add(
    components.transmission_line(
        id="right",
        length=1.0 * u.mm,
        rlgc=cpw,
        n_sections=1,
    )
)

`left` and `right` are separate native bodies. The next cell gives them
one same-net tap bus with three named attachments.

### Name the same-net bus and its three attachments

In [ ]:
input_bus = feedline.bus(id="input")
tap_bus = feedline.bus(id="tap")
output_bus = feedline.bus(id="output")

left_tap = tap_bus.tap(id="left_section")
right_tap = tap_bus.tap(id="right_section")
coupling_tap = tap_bus.tap(id="coupling")

The input, tap, and output buses plus these attachments provide every
endpoint handle required to place both bodies and expose the child
boundary. Their IDs and `.tap()` call order are not geometry commands.

### Place both bodies and expose the three public pins

In [ ]:
left_head_pin = left.pin("head", conductor="signal")
left_tail_pin = left.pin("tail", conductor="signal")
right_head_pin = right.pin("head", conductor="signal")
right_tail_pin = right.pin("tail", conductor="signal")

left_section = feedline.series(
    id="left_section",
    start=input_bus,
    elements=(
        left.between(left_head_pin, left_tail_pin),
    ),
    end=left_tap,
)
right_section = feedline.series(
    id="right_section",
    start=right_tap,
    elements=(
        right.between(right_head_pin, right_tail_pin),
    ),
    end=output_bus,
)
feedline_input_pin = feedline.expose_pin(id="input", at=input_bus)
feedline_tap_pin = feedline.expose_pin(id="tap", at=coupling_tap)
feedline_output_pin = feedline.expose_pin(id="output", at=output_bus)

The resulting public feedline handles are `feedline_input_pin`,
`feedline_tap_pin`, and `feedline_output_pin`; no reference-conductor
electrical handle was created. `coupling_tap` is an actual `TapRef` on
`tap_bus`, while `feedline_tap_pin` is the public `PinRef` returned by
`expose_pin(id="tap", ...)`.

The center tap is one electrical bus with distinct graphical
attachments. Each 1 mm `n_sections=1` segment is a finite pi
discretization, not exact distributed equivalence. Later 50 ohm root
Ports are retained loads, not a claim of exact match to this
approximately 48.99 ohm line.

### Compare automatic placement with an optional named-tap layout hint

Automatic placement derives from the declared structure. When a reader
needs a specific visual progression of the three named contacts,
`tap_order` is a pure presentation mapping: it neither changes the
shared `tap_bus` net nor adds a TapRef or changes series order.

In [ ]:
automatic_feedline = feedline_plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        show_parameter_values=True,
    )
)
automatic_feedline.show()

`automatic_feedline` is the unhinted drawing of the complete standalone
Plan. The next cell uses every explicit tap on `tap_bus` once to
describe only their geometric progression along that bus.

In [ ]:
tap_layout = SchematicLayout(
    tap_order={
        tap_bus: (
            left_tap,
            coupling_tap,
            right_tap,
        ),
    }
)
hinted_feedline = feedline_plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        show_parameter_values=True,
        layout=tap_layout,
    )
)
hinted_feedline.show()

`tap_order` is complete for the named taps of this bus. It controls
graphical contact order only; it does not infer physical flow from
source order or alter the line bodies, their endpoints, or the
electrical net.

## Lesson 12.3 — Sweep complete RLGC values

A whole RLGC value is one structured input with a fixed basis; its rows
and columns are not scalar sweep axes, and `n_sections` remains line
structure. The following baseline and sample use the existing `RLGC`
type and the same single-signal basis. This candidate supports response
points, not whole-matrix optimization, interpolation, or root
continuation across changed RLGC values.

In [ ]:
from scnsim import (
    ParameterDefinitions,
    ParameterSet,
    ParameterSpace,
    RLGCParameterSpec,
)

rlgc_inputs = ParameterDefinitions(id="feedline_rlgc")
rlgc_ref = rlgc_inputs.parameter(
    id="cpw",
    baseline=cpw,
    spec=RLGCParameterSpec(
        conductors=("signal",),
        reference_conductor="ground",
    ),
)
rlgc_sample = RLGC(
    conductors=("signal",),
    reference_conductor="ground",
    resistance_per_length=[[0.0]] * u.ohm / u.m,
    inductance_per_length=[[420.0]] * u.nH / u.m,
    conductance_per_length=[[0.0]] * u.S / u.m,
    capacitance_per_length=[[170.0]] * u.pF / u.m,
)
rlgc_space = ParameterSpace.points(
    (
        ParameterSet(),
        ParameterSet({rlgc_ref: rlgc_sample}),
    )
)

In [ ]:
parameterized_line_plan = CircuitPlan(id="parameterized_line")
parameterized_line = parameterized_line_plan.add(
    components.transmission_line(
        id="line",
        length=1.0 * u.mm,
        rlgc=rlgc_ref,
        n_sections=1,
    )
)
line_in = parameterized_line_plan.bus(id="input")
line_out = parameterized_line_plan.bus(id="output")
parameterized_line_plan.series(
    id="line_body",
    start=line_in,
    elements=(parameterized_line.between(
        parameterized_line.pin("head", conductor="signal"),
        parameterized_line.pin("tail", conductor="signal"),
    ),),
    end=line_out,
)
line_input_port = parameterized_line_plan.add_port(
    id="input",
    at=line_in,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
line_output_port = parameterized_line_plan.add_port(
    id="output",
    at=line_out,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

In [ ]:
from scnsim import (
    CircuitRun,
    DirectSolveSpec,
)

from IPython.display import display

parameterized_run = CircuitRun(
    plan=parameterized_line_plan,
    workspace="workspaces/rlgc-course",
)
response_spec = DirectSolveSpec(frequencies=[5.5, 6.0, 6.5] * u.GHz)
rlgc_sweep = parameterized_run.solve(
    parameterized_run.original,
    response_spec,
    parameters=rlgc_space,
)
rlgc_point = rlgc_sweep.points[1]
if rlgc_point.succeeded:
    display(rlgc_point.result)
else:
    display(rlgc_point.failure)

In [ ]:
selected_compiled = None
try:
    selected_compiled = parameterized_line_plan.render_schematic(
        CircuitDiagramSpec(representation="compiled"),
        parameters=rlgc_point.parameters,
    )
except SCNSimValidationError as error:
    if error.stage != "schematic_layout":
        raise
    display({
        "selected compiled schematic layout unavailable": str(error),
        "stage": error.stage,
        "evidence": dict(error.evidence),
    })
if selected_compiled is not None:
    display(selected_compiled.show())

In [ ]:
if selected_compiled is not None:
    display(selected_compiled.audit.show())

When available, the selected compiled audit shows the exact structured
input and expanded line at that stored Direct point. Chapters 15–16
introduce supported HB parameter spaces after their pump axes, drives,
cases, and truncation are established.

[Previous](11_review_customize_schematic.qmd) ·
[Next](13_compensate_probes.qmd)